In [1]:
import json
import re
import logging
from typing import Tuple, List, Dict
import torch
from enum import Enum
from dataclasses import dataclass, field

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class DiagramType(str, Enum):
    C4_CONTEXT = "c4_context"
    C4_CONTAINER = "c4_container"
    C4_COMPONENT = "c4_component"
    ERD = "erd"
    SEQUENCE = "sequence"
    BPMN = "bpmn"
    SCHEMATIC = "schematic"


class NodeType(str, Enum):
    PERSON = "person"
    SYSTEM = "system"
    CONTAINER = "container"
    COMPONENT = "component"
    DATABASE = "database"
    API_GATEWAY = "api_gateway"
    EXTERNAL_SYSTEM = "external_system"
    MESSAGE_BUS = "message_bus" # Added MESSAGE_BUS


class EdgeType(str, Enum):
    USES = "uses"
    CALLS = "calls"
    SENDS_TO = "sends_to"
    READS_FROM = "reads_from"
    WRITES_TO = "writes_to"


@dataclass
class Node:
    id: str
    name: str
    type: NodeType
    description: str = ""
    tags: List[str] = field(default_factory=list)
    properties: Dict[str, str] = field(default_factory=dict)

@dataclass
class Edge:
    source: str
    target: str
    type: EdgeType
    label: str = ""
    description: str = ""
    technology: str = ""

In [2]:
def _classify_intent(self, text: str) -> Tuple[DiagramType, float]:
    """
    Classify diagram type using T5 model with few-shot prompting.

    This method replaces keyword counting with AI-driven classification,
    enabling robust intent detection across varied phrasings and complex inputs.

    Args:
        text: Cleaned user input text

    Returns:
        Tuple of (DiagramType, confidence_score)
    """

    # Few-shot prompt with examples for each diagram type
    prompt = f"""Classify the following architecture description into ONE of these diagram types:

Diagram Types:
- c4_context: High-level system context, external systems, users
- c4_container: Applications, microservices, containers, APIs
- c4_component: Internal components, classes, modules
- erd: Database entities, tables, relationships, schema
- sequence: Process flows, interactions, timelines, steps
- bpmn: Business processes, workflows, procedures
- schematic: Circuits, electrical diagrams, wiring

Examples:
Input: "Show me the high-level architecture with external dependencies"
Output: c4_context

Input: "Create a system with 3 microservices and an API gateway"
Output: c4_container

Input: "Design a database schema for users and orders"
Output: erd

Input: "Show the sequence of API calls between services"
Output: sequence

Now classify this:
Input: "{text}"
Output:"""

    try:
        # Tokenize with proper length constraints
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            max_length=512,
            truncation=True
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Generate with sampling for better classification
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_length=20,  # Short output expected
                num_beams=4,
                temperature=0.7,
                do_sample=False,  # Deterministic for classification
                early_stopping=True
            )

        result_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()

        logger.info(f"[INTENT] Model output: '{result_text}'")

        # Parse model output to DiagramType enum
        # Handle various response formats
        result_text = result_text.replace("output:", "").strip()

        # Map common variations to enum values
        intent_mapping = {
            'c4_context': DiagramType.C4_CONTEXT,
            'c4 context': DiagramType.C4_CONTEXT,
            'context': DiagramType.C4_CONTEXT,
            'c4_container': DiagramType.C4_CONTAINER,
            'c4 container': DiagramType.C4_CONTAINER,
            'container': DiagramType.C4_CONTAINER,
            'microservice': DiagramType.C4_CONTAINER,
            'c4_component': DiagramType.C4_COMPONENT,
            'c4 component': DiagramType.C4_COMPONENT,
            'component': DiagramType.C4_COMPONENT,
            'erd': DiagramType.ERD,
            'entity relationship': DiagramType.ERD,
            'database': DiagramType.ERD,
            'sequence': DiagramType.SEQUENCE,
            'flow': DiagramType.SEQUENCE,
            'bpmn': DiagramType.BPMN,
            'workflow': DiagramType.BPMN,
            'process': DiagramType.BPMN,
            'schematic': DiagramType.SCHEMATIC,
            'circuit': DiagramType.SCHEMATIC
        }

        # Find best match
        intent = None
        confidence = 0.8  # High confidence for exact matches

        for key, diagram_type in intent_mapping.items():
            if key in result_text:
                intent = diagram_type
                break

        # Fallback: try direct enum value match
        if intent is None:
            for dt in DiagramType:
                if dt.value in result_text:
                    intent = dt
                    confidence = 0.9
                    break

        # Default fallback with low confidence
        if intent is None:
            logger.warning(f"[INTENT] Could not parse model output, defaulting to C4_CONTAINER")
            intent = DiagramType.C4_CONTAINER
            confidence = 0.3

        logger.info(f"[INTENT] Classified as {intent.value} (confidence: {confidence:.2f})")
        return intent, confidence

    except Exception as e:
        logger.error(f"[INTENT] Classification failed: {e}")
        # Safe fallback
        return DiagramType.C4_CONTAINER, 0.5

In [3]:
def _extract_structure_with_t5(self, text: str, intent: DiagramType) -> Dict:
    """
    Extract structured architecture using T5 with enhanced prompt engineering.

    Uses few-shot examples and implements repair mechanisms to handle
    conversational text wrapping JSON output.

    Args:
        text: User input text
        intent: Detected diagram type

    Returns:
        Dictionary with 'relationships' key containing list of connection dicts
    """

    # Enhanced prompt with JSON schema and examples
    prompt = f"""Extract technical architecture relationships as JSON.

Schema:
{{
  "relationships": [
    {{"from": "component_name", "to": "component_name", "type": "uses|calls|sends_to|reads_from|writes_to"}}
  ]
}}

Example 1:
Input: "API Gateway routes requests to UserService and OrderService"
JSON: {{"relationships": [{{"from": "API_Gateway", "to": "UserService", "type": "calls"}}, {{"from": "API_Gateway", "to": "OrderService", "type": "calls"}}]}}

Example 2:
Input: "Services read from PostgreSQL database"
JSON: {{"relationships": [{{"from": "Service", "to": "PostgreSQL", "type": "reads_from"}}]}}

Example 3:
Input: "User accesses the web application"
JSON: {{"relationships": [{{"from": "User", "to": "WebApp", "type": "uses"}}]}}

Now extract from:
Input: "{text}"
JSON:"""

    try:
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            max_length=512,
            truncation=True
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_length=256,
                num_beams=4,
                temperature=0.3,  # Lower temperature for structured output
                early_stopping=True
            )

        result_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        logger.info(f"[STRUCTURE] Raw model output: {result_text[:200]}")

        # Repair mechanism: extract JSON from conversational text
        json_result = _extract_and_repair_json(result_text)

        if json_result:
            # Validate schema
            if _validate_relationship_schema(json_result):
                logger.info(f"[STRUCTURE] Extracted {len(json_result.get('relationships', []))} relationships")
                return json_result
            else:
                logger.warning("[STRUCTURE] Schema validation failed")

        # Fallback: return empty structure
        logger.info("[STRUCTURE] Returning empty relationships (fallback)")
        return {"relationships": []}

    except Exception as e:
        logger.error(f"[STRUCTURE] Extraction failed: {e}")
        return {"relationships": []}

In [4]:
def _extract_and_repair_json(text: str) -> Dict:
    """
    Extract and repair JSON from model output that may contain conversational text.

    Args:
        text: Raw model output

    Returns:
        Parsed JSON dictionary or None
    """
    # Remove common conversational prefixes
    text = re.sub(r'^(here is|here\'s|the json is|output:?)\s*', '', text, flags=re.IGNORECASE)

    # Try direct parse first
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass

    # Extract JSON block using regex (handles markdown code blocks)
    json_pattern = r'```(?:json)?\s*(\{.*?\})\s*```'
    match = re.search(json_pattern, text, re.DOTALL)

    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass

    # Find first { to last } and try to parse
    try:
        start = text.index('{')
        end = text.rindex('}') + 1
        json_str = text[start:end]
        return json.loads(json_str)
    except (ValueError, json.JSONDecodeError):
        pass

    logger.warning("[REPAIR] Could not extract valid JSON from model output")
    return None

In [5]:
def _validate_relationship_schema(data: Dict) -> bool:
    """
    Validate that extracted JSON conforms to expected relationship schema.

    Args:
        data: Parsed JSON data

    Returns:
        True if valid, False otherwise
    """
    if not isinstance(data, dict):
        return False

    if 'relationships' not in data:
        return False

    relationships = data['relationships']
    if not isinstance(relationships, list):
        return False

    # Validate each relationship has required fields
    for rel in relationships:
        if not isinstance(rel, dict):
            return False
        if 'from' not in rel or 'to' not in rel:
            return False
        if not isinstance(rel['from'], str) or not isinstance(rel['to'], str):
            return False

    return True

In [6]:
def _infer_relationships(self, nodes: List[Node], intent: DiagramType) -> List[Edge]:
    """
    Infer relationships between nodes using T5 model instead of hardcoded patterns.

    Sends node information to the model and asks it to propose logical connections
    based on architectural best practices.

    Args:
        nodes: List of Node objects representing system components
        intent: Diagram type for context

    Returns:
        List of Edge objects representing inferred connections
    """

    if not nodes:
        return []

    # Prepare node information for the model
    node_descriptions = []
    for node in nodes:
        node_descriptions.append(f"- {node.name} (type: {node.type.value})")

    nodes_text = "\n".join(node_descriptions)

    # Available edge types for the model to use
    edge_types = ", ".join([et.value for et in EdgeType])

    # Construct prompt asking model to propose connections
    prompt = f"""Given these architectural components, propose logical connections based on standard best practices.

Components:
{nodes_text}

Available connection types: {edge_types}

Diagram context: {intent.value}

Architectural patterns to consider:
- Users access systems through gateways/interfaces
- API gateways route to backend services
- Services communicate via calls or message buses
- Services read/write to databases
- Components within same tier may call each other

Output format (JSON):
{{
  "connections": [
    {{"source": "component_name", "target": "component_name", "type": "uses|calls|sends_to|reads_from|writes_to", "label": "description"}}
  ]
}}

Propose connections:"""

    try:
        # Tokenize and generate
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            max_length=512,
            truncation=True
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_length=300,
                num_beams=5,
                temperature=0.4,
                early_stopping=True
            )

        result_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        logger.info(f"[RELATIONSHIPS] Model output: {result_text[:200]}")

        # Parse the model's response
        edges = _parse_connections_to_edges(result_text, nodes)

        logger.info(f"[RELATIONSHIPS] Inferred {len(edges)} connections using AI")
        return edges

    except Exception as e:
        logger.error(f"[RELATIONSHIPS] AI inference failed: {e}, using fallback")
        # Fallback to basic heuristics if AI fails
        return _fallback_relationship_inference(nodes)

In [7]:
def _parse_connections_to_edges(text: str, nodes: List[Node]) -> List[Edge]:
    """
    Parse model output into Edge objects with validation.

    Args:
        text: Model output text
        nodes: List of available nodes to validate references

    Returns:
        List of valid Edge objects
    """
    edges = []

    # Extract JSON from response
    json_data = _extract_and_repair_json(text)

    if not json_data or 'connections' not in json_data:
        logger.warning("[RELATIONSHIPS] Could not parse connections from model output")
        return edges

    # Create node lookup for validation
    node_map = {node.name.lower(): node.id for node in nodes}
    node_map.update({node.id.lower(): node.id for node in nodes})

    # Parse each connection
    for conn in json_data.get('connections', []):
        try:
            source_name = conn.get('source', '').strip()
            target_name = conn.get('target', '').strip()
            edge_type_str = conn.get('type', 'uses').strip().lower()
            label = conn.get('label', '')

            # Find matching nodes (fuzzy matching)
            source_id = _find_matching_node_id(source_name, node_map, nodes)
            target_id = _find_matching_node_id(target_name, node_map, nodes)

            if not source_id or not target_id:
                logger.warning(f"[RELATIONSHIPS] Could not find nodes for {source_name} -> {target_name}")
                continue

            # Validate edge type
            edge_type = EdgeType.USES  # Default
            for et in EdgeType:
                if et.value == edge_type_str or et.value.replace('_', '') == edge_type_str.replace('_', ''):
                    edge_type = et
                    break

            # Create edge
            edge = Edge(
                source=source_id,
                target=target_id,
                type=edge_type,
                label=label if label else edge_type.value
            )
            edges.append(edge)

        except Exception as e:
            logger.warning(f"[RELATIONSHIPS] Failed to parse connection: {e}")
            continue

    return edges

In [8]:
def _find_matching_node_id(name: str, node_map: Dict[str, str], nodes: List[Node]) -> str:
    """
    Find node ID using fuzzy matching.

    Args:
        name: Component name from model output
        node_map: Dictionary mapping names to IDs
        nodes: List of all nodes

    Returns:
        Node ID if found, None otherwise
    """
    name_lower = name.lower().replace(' ', '_').replace('-', '_')

    # Direct lookup
    if name_lower in node_map:
        return node_map[name_lower]

    # Fuzzy matching - check if name is contained in any node name
    for node in nodes:
        node_name_clean = node.name.lower().replace(' ', '_').replace('-', '_')
        if name_lower in node_name_clean or node_name_clean in name_lower:
            return node.id

    return None

In [9]:
def _fallback_relationship_inference(nodes: List[Node]) -> List[Edge]:
    """
    Fallback heuristic-based relationship inference if AI fails.

    Uses simple rules as a safety net.

    Args:
        nodes: List of Node objects

    Returns:
        List of Edge objects
    """
    edges = []

    # Categorize nodes by type
    users = [n for n in nodes if n.type == NodeType.PERSON]
    gateways = [n for n in nodes if n.type == NodeType.API_GATEWAY]
    services = [n for n in nodes if n.type == NodeType.COMPONENT]
    databases = [n for n in nodes if n.type == NodeType.DATABASE]

    # Simple pattern: User -> Gateway -> Services -> Database
    if users and gateways:
        edges.append(Edge(users[0].id, gateways[0].id, EdgeType.USES, "accesses"))

    if gateways and services:
        for service in services[:3]:  # Limit to avoid too many edges
            edges.append(Edge(gateways[0].id, service.id, EdgeType.CALLS, "routes to"))

    if services and databases:
        for service in services[:3]:
            edges.append(Edge(service.id, databases[0].id, EdgeType.READS_FROM, "reads/writes"))

    logger.info(f"[RELATIONSHIPS] Fallback heuristics created {len(edges)} edges")
    return edges

In [10]:
from pathlib import Path
import uuid

@dataclass
class UnifiedGraphISR:
    metadata: Dict[str, str] = field(default_factory=dict)
    nodes: List[Node] = field(default_factory=list)
    edges: List[Edge] = field(default_factory=list)

In [11]:
import zlib
import base64
import requests
from IPython.display import Image, display

class C4DiagramRenderer:
    """
    Renders UnifiedGraphISR to C4 diagrams using PlantUML.
    Includes support for displaying images directly in Colab.
    Updated to fix syntax errors caused by newlines in titles.
    """

    def __init__(self, output_dir: str = "./generated_diagrams"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True, parents=True)

    def render(self, isr: UnifiedGraphISR, output_filename: str = None) -> Dict:
        """
        Render ISR to C4 diagram source and display it.
        """
        if output_filename is None:
            output_filename = f"diagram_{uuid.uuid4().hex[:8]}"

        # Generate PlantUML source
        puml_content = self._generate_plantuml_c4(isr)

        # Save .puml file locally
        puml_path = self.output_dir / f"{output_filename}.puml"
        with open(puml_path, 'w') as f:
            f.write(puml_content)

        print(f"[RENDER] Saved PlantUML source: {puml_path}")

        # Display the image in the notebook
        print("[RENDER] Rendering image...")
        self.display_diagram(puml_content)

        return {
            "puml_path": str(puml_path),
            "success": True,
            "errors": []
        }

    def display_diagram(self, puml_content: str):
        """
        Encodes PlantUML source and fetches the image from the official server.
        """
        try:
            # 1. Compress and encode the PlantUML code for the URL
            zlibbed_str = zlib.compress(puml_content.encode('utf-8'))
            compressed_string = zlibbed_str[2:-4]
            encoded_string = base64.b64encode(compressed_string).decode('utf-8')

            # PlantUML uses a custom base64 mapping
            mapping = str.maketrans(
                'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/',
                '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz-_'
            )
            encoded_url_string = encoded_string.translate(mapping)

            # 2. Construct the URL
            url = f"https://www.plantuml.com/plantuml/png/{encoded_url_string}"

            # 3. Display the image
            display(Image(url=url))

        except Exception as e:
            print(f"[RENDER] Could not display image: {e}")

    def _generate_plantuml_c4(self, isr: UnifiedGraphISR) -> str:
        """Generate PlantUML C4 diagram source code"""
        lines = [
            "@startuml",
            "!include https://raw.githubusercontent.com/plantuml-stdlib/C4-PlantUML/master/C4_Container.puml",
            "",
            "LAYOUT_WITH_LEGEND()",
            ""
        ]

        # FIX: Sanitize title to remove special chars AND NEWLINES that break PlantUML
        raw_title = isr.metadata.get('cleaned_input', 'Architecture')

        # Replace newlines with spaces, remove parens, and strip whitespace
        safe_title = raw_title.replace('\n', ' ').replace('\r', '').replace('(', '').replace(')', '').replace('[', '').replace(']', '').strip()

        # Truncate to 60 chars
        title = f"System: {safe_title[:60]}..."

        lines.append(f"title {title}")
        lines.append("")

        # Add nodes
        node_type_map = {
            NodeType.PERSON: "Person",
            NodeType.SYSTEM: "System",
            NodeType.CONTAINER: "Container",
            NodeType.COMPONENT: "Container",
            NodeType.DATABASE: "ContainerDb",
            NodeType.MESSAGE_BUS: "ContainerQueue",
            NodeType.API_GATEWAY: "Container",
            NodeType.EXTERNAL_SYSTEM: "System_Ext"
        }

        for node in isr.nodes:
            c4_type = node_type_map.get(node.type, "Container")
            tech = node.technology or ""
            desc = node.description or f"A {node.type.value}"
            lines.append(f'{c4_type}({node.id}, "{node.name}", "{desc}", "{tech}")')

        lines.append("")

        # Add relationships
        for edge in isr.edges:
            label = edge.label or edge.type.value
            tech = edge.technology or ""
            lines.append(f'Rel({edge.source}, {edge.target}, "{label}", "{tech}")')

        lines.append("")
        lines.append("@enduml")

        return "\n".join(lines)

In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [13]:
class DiagramGenerationPipeline:
    def __init__(self, model_name: str = "google/flan-t5-large", device: str = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"[INIT] Loading model: {model_name} on {self.device}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(self.device)
        self.renderer = None

    def generate_diagram(self, user_input: str):
        logger.info(f"[PIPELINE] Starting diagram generation for: {user_input[:100]}...")

        diagram_type, confidence = self._classify_intent(user_input)
        logger.info(f"[PIPELINE] Classified intent: {diagram_type.value} with confidence {confidence:.2f}")

        extracted_data = self._extract_structure_with_t5(user_input, diagram_type)
        initial_relationships = extracted_data.get('relationships', [])
        logger.info(f"[PIPELINE] Extracted {len(initial_relationships)} initial relationships.")

        nodes = self._create_nodes_from_relationships(initial_relationships, diagram_type)

        if not nodes:
            print("\n❌ WARNING: No components were found! The diagram will be empty.")
            print("   Possible Cause: The model couldn't understand the input structure.")
            print("   Solution: Ensure you are using 'flan-t5-large' and try simplifying the prompt.\n")

        logger.info(f"[PIPELINE] Created {len(nodes)} nodes.")

        inferred_edges = self._infer_relationships(nodes, diagram_type)
        logger.info(f"[PIPELINE] Inferred {len(inferred_edges)} additional edges.")

        isr = UnifiedGraphISR(
            metadata={
                "user_input": user_input,
                "cleaned_input": user_input,
                "diagram_type": diagram_type.value
            },
            nodes=nodes,
            edges=inferred_edges
        )

        if self.renderer:
            logger.info("[PIPELINE] Rendering diagram...")
            render_result = self.renderer.render(isr)
            logger.info(f"[PIPELINE] Diagram rendered: {render_result.get('puml_path')}")
        else:
            logger.warning("[PIPELINE] No renderer set. Skipping diagram rendering.")

        return isr

    def _create_nodes_from_relationships(self, relationships: List[Dict], intent: DiagramType) -> List[Node]:
        node_names = set()
        for rel in relationships:
            node_names.add(rel['from'])
            node_names.add(rel['to'])

        nodes = []
        for name in node_names:
            node_id = name.lower().replace(' ', '_').replace('-', '_')
            node_type = NodeType.SYSTEM

            if "user" in name.lower() or "person" in name.lower():
                node_type = NodeType.PERSON
            elif "database" in name.lower() or "db" in name.lower() or "sql" in name.lower() or "postgres" in name.lower():
                node_type = NodeType.DATABASE
            elif "api gateway" in name.lower() or "gateway" in name.lower():
                node_type = NodeType.API_GATEWAY
            elif "service" in name.lower() or "app" in name.lower() or "microservice" in name.lower():
                node_type = NodeType.CONTAINER if intent == DiagramType.C4_CONTAINER else NodeType.COMPONENT
            elif "external" in name.lower() or "third-party" in name.lower():
                node_type = NodeType.EXTERNAL_SYSTEM

            nodes.append(Node(id=node_id, name=name, type=node_type, description=f"A {node_type.value}: {name}"))
        return nodes

DiagramGenerationPipeline._classify_intent = _classify_intent
DiagramGenerationPipeline._extract_structure_with_t5 = _extract_structure_with_t5
DiagramGenerationPipeline._extract_and_repair_json = staticmethod(_extract_and_repair_json)
DiagramGenerationPipeline._validate_relationship_schema = staticmethod(_validate_relationship_schema)
DiagramGenerationPipeline._infer_relationships = _infer_relationships
DiagramGenerationPipeline._parse_connections_to_edges = staticmethod(_parse_connections_to_edges)
DiagramGenerationPipeline._find_matching_node_id = staticmethod(_find_matching_node_id)
DiagramGenerationPipeline._fallback_relationship_inference = staticmethod(_fallback_relationship_inference)

In [14]:
# Initialize pipeline (if not already done)
pipeline = DiagramGenerationPipeline()
pipeline.renderer = C4DiagramRenderer()

# Input: E-commerce system using Kafka
user_input = """
Design an event-driven system where an Order Service publishes 'OrderCreated' events to a Kafka Message Bus.
A Shipping Service consumes these events to schedule delivery, and a Notification Service also consumes them to send emails to the User.
"""

pipeline.generate_diagram(user_input)

[INIT] Loading model: google/flan-t5-large on cpu...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



❌ WARNING: No components were found! The diagram will be empty.
   Possible Cause: The model couldn't understand the input structure.
   Solution: Ensure you are using 'flan-t5-large' and try simplifying the prompt.

[RENDER] Saved PlantUML source: generated_diagrams/diagram_9f87e0ec.puml
[RENDER] Rendering image...


UnifiedGraphISR(metadata={'user_input': "\nDesign an event-driven system where an Order Service publishes 'OrderCreated' events to a Kafka Message Bus. \nA Shipping Service consumes these events to schedule delivery, and a Notification Service also consumes them to send emails to the User.\n", 'cleaned_input': "\nDesign an event-driven system where an Order Service publishes 'OrderCreated' events to a Kafka Message Bus. \nA Shipping Service consumes these events to schedule delivery, and a Notification Service also consumes them to send emails to the User.\n", 'diagram_type': 'bpmn'}, nodes=[], edges=[])

In [15]:
complex_input = """
Design a Ride Sharing System where:
1. A Passenger uses the Mobile App to request a ride.
2. The App calls the Matchmaking Service to find a driver.
3. The Matchmaking Service tracks location using a Redis Cache.
4. After the ride, the Payment Service charges the Passenger and pays the Driver.
"""
pipeline.generate_diagram(complex_input)


❌ WARNING: No components were found! The diagram will be empty.
   Possible Cause: The model couldn't understand the input structure.
   Solution: Ensure you are using 'flan-t5-large' and try simplifying the prompt.

[RENDER] Saved PlantUML source: generated_diagrams/diagram_3f9b4d35.puml
[RENDER] Rendering image...


UnifiedGraphISR(metadata={'user_input': '\nDesign a Ride Sharing System where:\n1. A Passenger uses the Mobile App to request a ride.\n2. The App calls the Matchmaking Service to find a driver.\n3. The Matchmaking Service tracks location using a Redis Cache.\n4. After the ride, the Payment Service charges the Passenger and pays the Driver.\n', 'cleaned_input': '\nDesign a Ride Sharing System where:\n1. A Passenger uses the Mobile App to request a ride.\n2. The App calls the Matchmaking Service to find a driver.\n3. The Matchmaking Service tracks location using a Redis Cache.\n4. After the ride, the Payment Service charges the Passenger and pays the Driver.\n', 'diagram_type': 'bpmn'}, nodes=[], edges=[])